In [ ]:
from selenium import webdriver # 크롤러 역할
from selenium.webdriver.common.by import By # 크롤러가 어떤 방식으로 값을 찾아오게 할 것인지 결정
from selenium.webdriver.chrome.service import Service # 크롤러에게 로컬컴퓨터 내 크롬브라우저 드라이브 일치 & 설치 지원
from selenium.webdriver.chrome.options import Options # 크롤러가 어떤 환경에서 어떤 방식으로 크롤링 하도록 할지 옵션 설정
from webdriver_manager.chrome import ChromeDriverManager # 실제 드라이브 설치 역할

import pandas as pd
import time # 특정 사이트 접속 후 스크립트 기반의 코드를 서버로부터 가져올 때까지 시간을 벌어주는 역할
import re # regular expression
service = Service(ChromeDriverManager().install())
options = Options()

options.add_argument("--headless") # 가상브라우저를 현재 내 로컬컴퓨터에 나타나지 않도록 할 때
options.add_argument("--disable-gpu") # 가상브라우저를 사용하지 않기 때문에 굳이 gpu 메모리를 사용하지 않겠다는 의미
options.add_argument("--disable-dev-shm-usage") # 큰 데이터의 크롤링 시, 데이터를 개발자도구 내 임시메모리 공간을 사용 x (로컬공간) 
options.add_argument("--window-size=1920,1080") # 크롤러의 웹 브라우저 사이즈 세팅
options.add_argument("--start-maximized") # 주변 환경요소와 무관하게 브라우저 사이즈를 무조건 크게
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36") # 브라우저 식별 정보 = 우리가 정상적인 웹사이트 방문자
options.add_argument("--lang=ko_KR") # 접속 및 방문하 브라우저의 기본세팅 언어
options.add_argument("--no-sandbox") # 규제 적용 방지

URL = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
time.sleep(3)

# 유틸리티 함수
def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0

# 베스트 버튼 클릭
best_button = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_button.click()
time.sleep(3)

# 상품카드 정보 크롤링
items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print(f"1. ✅ 찾은 상품 수: {len(items)}개")

# 상품 수집데이터 저장용 리스트 생성
products = list() # []

# 상품별 상세데이터 수집
for idx, item in enumerate(items, 1) :
    # 상품 랭킹 번호 수집
    try :
        ranking_no = int(item.find_element(By.CSS_SELECTOR, "span.rank").text)
    except :
        ranking_no = None

    # 브랜드 이름 수집
    try :
        brand_name = item.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
    except :
        brand_name = None

    # 상품 이름 수집
    try :
        product_name = item.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()
    except :
        product_name = None
    
    # 판매가격 (*소비자가, 할인가, 할인률) 수집
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    # 소비자가
    try :
        original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text)
    except :
        original_price = None

    # 할인가
    try :
        sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text)
    except :
        sale_price = None
        
    # 할인률
    try :
        discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text)
    except :
        discount_rate = 0

    try :
        stats = item.find_element(By.CSS_SELECTOR, "span.stats")
    
        # 평점
        try :
            rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text)
        except :
            rating = None
    
        # 리뷰수
        try :
            review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text)
        except :
            review_count = 0
    
    
        # 좋아요수
        try :
            like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like span.text.cnt").text)
        except :
            like_count = 0
    
    except :
        rating = None
        review_count = 0
        like_count = 0

    # 상품상세페이지 url
    img_url = item.find_element(By.CSS_SELECTOR, "span.img img").get_attribute("src")

    match = re.search(r"/(\d{9})[_.]", img_url)

    if match :
        product_id = match.group(1)
        product_url = f"https://www.wconcept.co.kr/Product/{product_id}"
    else :
        product_id = None
        product_url = None
            
    products.append({
        "ranking_no": ranking_no,
        "brand_name": brand_name,
        "product_name": product_name,
        "original_price": original_price,
        "sale_price": sale_price,
        "discount_rate": discount_rate,
        "rating": rating,
        "review_count": review_count,
        "like_count": like_count,
        "product_id": product_id,
        "product_url": product_url,
        "img_url": img_url,
        "category_depth1": None,
        "category_depth2": None,
        "category_depth3": None,
        "category_depth4": None
    })
   
print("2. ✅ wconcept 상품기본 정보수집 완료!")

# 각 상품별 리뷰수집 데이터 저장용 리스트
all_reviews = list() # []

#for product in products[:1]
for product in products :

    product_url = product["product_url"]

    # 상품 상세페이지 진입단계
    # --------------------------------------------------------
    # 상품 URL이 없으면 상세페이지 접근 불가
    # --------------------------------------------------------

    if product_url is None:
        print("상품 URL 없음 → SKIP")
        continue

    # --------------------------------------------------------
    # 상품 상세페이지 진입
    # --------------------------------------------------------

    driver.get(product_url)
    time.sleep(10)

    # --------------------------------------------------------
    # 카테고리 수집
    # --------------------------------------------------------
    
    try:
        category_elements = driver.find_elements(
            By.CSS_SELECTOR,
            "#container > .pdt > #prdLocaiton > li > a, #container > .pdt > #prdLocaiton > li > button"
        )
    
        category_list = []
    
        for element in category_elements:
            text = element.text.strip()
    
            if text:
                category_list.append(text)
    
        # HOME 제거
        if category_list and category_list[0] == "HOME":
            category_list = category_list[1:]
    
        print("수집된 카테고리 :", category_list)
    
    except Exception as e:
    
        print("카테고리 수집 실패 :", e)
    
        category_list = []

    # --------------------------------------------------------
    # depth 별 분리
    # --------------------------------------------------------

    product["category_depth1"] = category_list[0] if len(category_list) > 0 else None
    product["category_depth2"] = category_list[1] if len(category_list) > 1 else None
    product["category_depth3"] = category_list[2] if len(category_list) > 2 else None
    product["category_depth4"] = category_list[3] if len(category_list) > 3 else None

    # --------------------------------------------------------
    # 리뷰가 없는 상품이면 카테고리만 저장하고 다음 상품으로
    # --------------------------------------------------------

    if product["review_count"] == 0 :
        print("리뷰 없음 → 카테고리만 수집")
        continue

    try : 
        review_button = driver.find_element(By.CSS_SELECTOR, "#reviewCnt1")
        review_button.click()
    
        time.sleep(3)
        # print("2-1. ✅ 리뷰버튼 클릭 완료!")
    except :
        # print("2-2. ✅ 리뷰버튼이 없는 상품입니다 -> SKIP")
        continue

    print("3. ✅ wconcept 상품상세페이지 진입 완료!")
    for page_no in range(1, 4) :
        if page_no > 1 :
            page_buttons = driver.find_elements(By.CSS_SELECTOR, f"ul#reviewPageNavigation a[title='{page_no}']")

            if len(page_buttons) == 0 :
                print(f"{page_no}페이지가 없습니다.")
                print("리뷰 수집을 종료합니다.")

                break

            page_buttons[0].click()

            time.sleep(2)
                
        rows = driver.find_elements(By.CSS_SELECTOR, "tr")

        # 진짜 리뷰 정보값을 가지고 있는 tr만 저장할 목적의 리스트
        review_items = list()
    
        for row in rows :
            review_text_elements = row.find_elements(By.CSS_SELECTOR, "p.pdt_review_text")
    
            if len(review_text_elements) > 0 :
                review_items.append(row)

        print(f"{page_no}) 페이지 리뷰 수: {len(review_items)}개")
    
        for review_index, review in enumerate(review_items, 1) :

            review_no = (page_no - 1) * 6 + review_index
            
            # 리뷰 작성자
            reviewer = review.find_element(By.CSS_SELECTOR, "p.product_review_info_right > em").text.strip()
    
            # 리뷰 작성일
            review_date = review.find_element(By.CSS_SELECTOR, "p.product_review_info_right > span").text.strip()
    
            # 구매 옵션정보
            try :
                purchase_text = review.find_element(By.CSS_SELECTOR, "div.pdt_review_option span").text.strip()
                purchase_option = re.sub(r"^구매옵션\s*:\s*", "", purchase_text)
                purchase_option = purchase_option.rstrip(",").strip()

                if purchase_option == "" :
                    purchase_option = None
            except :
                purchase_option = None
    
            # 리뷰 본문
            review_text = review.find_element(By.CSS_SELECTOR, "p.pdt_review_text").text.strip()
            review_text = re.sub(r"\s+", " ", review_text)
            
            # 리뷰 별점
            star = review.find_element(By.CSS_SELECTOR, "div.star-grade > strong")
            style = star.get_attribute("style")
            match = re.search(r"width:\s*([0-9]+)%", style)
    
            if match :
                star_percent = float(match.group(1))
                review_rating = star_percent / 20
            else :
                review_rating = None
    
            # 리뷰 상세 평가 데이터 저장을 위한 딕셔너리 정의
            evaluations = dict() # | {}
    
            # 리뷰 상세 평가
            try :
                evaluation_items = review.find_elements(By.CSS_SELECTOR, "ul.product_review_evaluation > li")
                for evaluation in evaluation_items :
                    evaluation_type = evaluation.find_element(By.CSS_SELECTOR, "strong").text.strip()
                    evaluation_value = evaluation.find_element(By.CSS_SELECTOR, "em").text.strip()
                    evaluations[evaluation_type] = evaluation_value
            except :
                continue
    
            # 리뷰 이미지 저장용 리스트
            review_images = list()
    
            # 리뷰 이미지
            images = review.find_elements(By.CSS_SELECTOR, "ul.pdt_review_photo img")
        
            for image in images :
                image_url = image.get_attribute("src")
                review_images.append(image_url)

            review_data = {
                "product_id": product["product_id"],
                "brand_name": product["brand_name"],
                "product_name": product["product_name"],
                "product_url": product["product_url"],
                "review_page": page_no,
                "review_no": review_no,
                "reviewer": reviewer,
                "review_date": review_date,
                "purchase_option": purchase_option,
                "review_text": review_text,
                "review_rating": review_rating,
                "evaluations": evaluations,
                "review_images": review_images                
            }

            all_reviews.append(review_data)
            
product_df= pd.DataFrame(products)
review_df = pd.DataFrame(all_reviews)            
print("4. ✅ wconcept 전체리뷰수집 완료")

In [ ]:
import re
import requests
import pandas as pd
import time

NAVER_CLIENT_ID = ""
NAVER_CLIENT_SECRET = ""

NAVER_BLOG_API_URL = (
    "https://naverapihub.apigw.ntruss.com/search/v1/blog"
)

headers = {
    "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
    "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET
}

# ============================================================
# 텍스트 정제 함수
# ============================================================

def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"&amp;", "&", text)
    return text.strip()

# ============================================================
# 브랜드 목록
# ============================================================

brand_names = (
    product_df["brand_name"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("검색할 전체 브랜드 수:", len(brand_names))

# ============================================================
# 전체 블로그 데이터 저장
# ============================================================

all_blog_posts = []

# ============================================================
# 브랜드별 수집
# ============================================================

for brand_index, brand_name in enumerate(brand_names, 1):
    print(f"[{brand_index}/{len(brand_names)}] {brand_name} 수집 시작")

    # ========================================================
    # 브랜드당 최대 1,000개
    #
    # 1
    # 101
    # 201
    # ...
    # 901
    # ========================================================

    for start in range(1, 1001, 100):
        print(f"start={start} 검색 중...")
        params = {
            "query": brand_name,
            "display": 100, # 한 번 호출에 최대 100개
            "start": start,  # 1, 101, 201 ...
            "sort": "date",
            "format": "json"
        }

        try:
            response = requests.get(
                NAVER_BLOG_API_URL,
                headers=headers,
                params=params,
                timeout=10
            )
            
            # ------------------------------------------------
            # 정상 응답
            # ------------------------------------------------

            if response.status_code == 200:
                data = response.json()
                items = data.get("items", [])

                # 검색 결과가 더 이상 없으면 종료
                if len(items) == 0:
                    print("더 이상 검색 결과 없음")
                    break

                for item in items:

                    blog_data = {
                        "search_brand": brand_name,
                        "title": clean_text(item.get("title", "")),
                        "link": item.get("link",""),
                        "description": clean_text(item.get("description","")),
                        "bloggername": item.get("bloggername", ""),
                        "bloggerlink":item.get("bloggerlink", ""),
                        "postdate":item.get("postdate", "")
                    }

                    all_blog_posts.append(blog_data)

                print(f"{len(items)}개 수집")

            # ------------------------------------------------
            # API 오류
            # ------------------------------------------------

            else:
                print(f"API 오류: {response.status_code}")
                print(response.text)
                break


        except Exception as e:
            print(f"API 호출 중 오류 발생: {e}")
            break

        # 너무 연속적으로 요청하지 않도록 약간의 간격
        time.sleep(0.1)

# ============================================================
# DataFrame 생성
# ============================================================

blog_df = pd.DataFrame(all_blog_posts)

# ============================================================
# 날짜 형식 변환
# ============================================================

if not blog_df.empty:
    blog_df["postdate"] = pd.to_datetime(
        blog_df["postdate"],
        format="%Y%m%d",
        errors="coerce"
    )


# ============================================================
# 중복 URL 제거
#
# 같은 글이 여러 검색 결과에서 중복될 수 있으므로
# 필요할 경우 사용
# ============================================================

# blog_df = (
#     blog_df
#     .drop_duplicates(
#         subset=[
#             "search_brand",
#             "link"
#         ]
#     )
#     .reset_index(
#         drop=True
#     )
# )


# ============================================================
# 최종 결과
# ============================================================

print("=" * 70)
print("수집 완료")
print("브랜드 수:", len(brand_names))
print("블로그 게시물 수:", len(blog_df))
print("=" * 70)

In [ ]:
# product_df : 상품 200개의 데이터 저장 -> 코드 + 취합 : 10분
# review_df : 각 상품별 리뷰 데이터 저장 (최소 0개부터 최대 18개까지 리뷰) -> 코드 + 취합 : 60분
# blog_df : 위(product_df)에서 수집한 각 브랜드별 네이버 블로그 내 컨텐츠 1000개씩 데이터 저장

# 왜 우리는 위 데이터가 필요했을까? -> Gen AI 왜 요청해야할까? | 직접해야할까?
# 1) wconcept 플랫폼 현재 어떤 상품 및 브랜드를 잘 & 많이 판매하고 있을까? -> wconcept 마케터라면 | 우리의 경쟁업체 wconcept : 3C, STP, SWOT
# 2) 퍼포먼스 마케팅 진행한다면, 외부 집행할 광고컨텐츠 및 컨셉, 어떤 방향 -> 내부에서 소비자들의 니즈, 관심도 : Ads Admin
# 3) 네이버 블로그 API 데이터 활용 -> IMC (통합마케팅전략), 바이럴 마케팅 KPI 확인

# 컴퓨터를 껐다가, 다시 켤 때마다 60분을 매번 소비?
# 만약, 나 + 팀원 3명 => 60분 => 240분 (매일 소진) 480분 시간 소비

# 아무리 좋은 데이터를 목적에 맞춰서 잘 수집을 했어도
# 해당 데이터를 잘 저장해놓지 못하면 아무 의미가 없다!!
# 데이터 : 수집 -> 저장

# 저장할 수 있는 양식.형태 등
# 각 양식 중 가장 효율적.가성비 포맷
# 가장 효율적.가성비 포맷 다룰 수 있는 능력.기술 => Database -> Table -> Data -> Search
# Oracle(난이도 상), MySQL(난이도 중.하) => SQL 쿼리언어의 문법

# 1차 저장 : Raw Data(Crawling) -> CSV, JSONL -> MySQL
# 전제조건 : Raw Data(product_df, review_df, blog_df) 3개의 파일을 같은 부모폴더 아래에 wconcept_data 폴더 생성 후 파일을 생성

In [2]:
import os # opertating system (폴더 신규생성, 조회, 자료삽입)
import json # 파이썬 내 자료구조 중 하나, 딕셔너리(객체)의 모양을 그대로 유지한 상태로 문자열로 변환하거나, 혹은 그 반대의 기능 구현
from datetile import datetime

SAVE_DIR = "./wconcept_data"

os.makedirs(SAVE_DIR, exist_ok=True)
collected_at = datetime.now().strftime("%Y%m%d_%H%M%S") # 20260916_110823

product_file = f"{SAVE_DIR}/wconcept_products_{collected_at}.csv"
# ./wconcept_data/wconcept_products_20260916_110823.csv

product_df.to_csv(product_file, index=False, encoding="utf-8-sig")
# 1:1 => 브랜드명 : "나이키"
# 1:N => 상세평가 : '{"사이즈": "적당함", "색상": "비슷함"}'

review_save_df = review_df.copy()

review_save_df["evaluations"] = (
    review_save_df["evaluations"]
    .apply(
        lambda x : json.dumps(
            x, ensure_ascii=False
        ) 
        if isinstance(x, dict)
        else "{}"
    )
)

review_save_df["review_images"] = (
    review_save_df["review_images"]
    .apply(
        lambda x : json.dumps(
            x, ensure_ascii=False
        ) 
        if isinstance(x, list)
        else "[]"
    )
)

review_file = f"{SAVE_DIR}/wconcept_reviews_{collected_at}.csv"
review_save_df.to_csv(review_file, index=False, encoding="utf-8-sig")

review_json_file = f"{SAVE_DIR}/wconcept_reviews_{collected_at}.jsonl"

review_df.to_json(review_json_file, orient="records", lines=True, force_ascii=False)

# brands products review_text (필드명)
# sfsdf   sdfsdf    sdfsdf (record = raw = 행)
# sdfsdf  adas      asdasd
# asdasd  adasd     adsasd

# {"brands": "sfsdf💚", "products": "sdfsdf", "review_text": "sdfsdf"}
# {"brands": "sdfsdf", "products": "adas", "review_text": "asdasd"}
# {"brands": "asdasd", "products": "adasd", "review_text": "adsasd"}


# 조건(문)
# if isinstance(x, dict) :
#     lambda x : json.dumps(x, ensure_ascii=False)
# else :
#     "{}"

# 람다식 = 익명함수
# def func1(a, b) :
#     return a + b
# 함수구(문) -> 문장 (복문)
# 문 -> 식 : 단문으로 완성


# "evaluations"
# a {"사이즈": "적당함", "색상": "비슷함"}
# b {"사이즈": "적당함", "색상": "비슷함"}
# c {"사이즈": "적당함", "색상": "비슷함"}

blog_csv_file = f"{SAVE_DIR}/wconcept_blogs_{collected_at}.csv"

blog_df.to_csv(blog_csv_file, index=False, encoding="utf-8-sig")

SyntaxError: invalid syntax (1664247266.py, line 42)

In [16]:
import pandas as pd

product_df = pd.read_csv("./wconcept_data/wconcept_products_20260913_234742.csv", dtype={"product_id": str})
review_df = pd.read_csv("./wconcept_data/wconcept_reviews_20260913_234742.csv", dtype={"product_id": str})
review_jsonl_df = pd.read_json("./wconcept_data/wconcept_reviews_20260913_234742.jsonl", lines=True)
blog_df = pd.read_csv("./wconcept_data/wconcept_blogs_20260914_001928.csv")

In [14]:
review_df["evaluations"] = (
    review_df["evaluations"]
    .apply(
        lambda x : json.loads(x) 
        if pd.notna(x) # na = not available 
        else {}
    )
)

review_df["review_images"] = (
    review_df["review_images"]
    .apply(
        lambda x : json.loads(x) 
        if pd.notna(x) # na = not available 
        else []
    )
)

blog_df["postdate"] = pd.to_datetime(
    blog_df["postdate"], errors="coerce"
)

In [17]:
review_jsonl_df

,product_id,brand_name,product_name,product_url,review_page,review_no,reviewer,review_date,purchase_option,review_text,review_rating,evaluations,review_images
0,308886954,틸아이다이,[단 하루!] [헤이가가 Pick]Wool cashmere V neck knit s...,https://www.wconcept.co.kr/Product/308886954,1,1,ye******,2026.09.13,IVORY_FREE,예뻐요 빨리 추워져서 입고 나가고 싶어요 얼굴도 환하게 해주고 굿굿입니다,5,"{'사이즈': '적당함', '색상': '같음', '소재': '같음'}",[https://media.wconcept.co.kr/review/308886954...
1,308886954,틸아이다이,[단 하루!] [헤이가가 Pick]Wool cashmere V neck knit s...,https://www.wconcept.co.kr/Product/308886954,1,2,wc******,2026.09.13,IVORY_FREE,따둣해서 지금 입기좋아요 속에 옷은 신경써야될듯,5,"{'사이즈': '큼', '색상': '다름', '소재': '다름'}",[]
2,308886954,틸아이다이,[단 하루!] [헤이가가 Pick]Wool cashmere V neck knit s...,https://www.wconcept.co.kr/Product/308886954,1,3,he******,2026.09.12,IVORY_FREE,팔이 진짜 길긴 길어요ㅎㅎ 제 덩치에는 여리여리 한 느낌은 없지만 맨살에 괜찮고 잘...,4,"{'사이즈': '적당함', '색상': '같음', '소재': '비슷함'}",[https://media.wconcept.co.kr/review/308886954...
3,308886954,틸아이다이,[단 하루!] [헤이가가 Pick]Wool cashmere V neck knit s...,https://www.wconcept.co.kr/Product/308886954,1,4,wc******,2026.09.11,IVORY_FREE,아직 입고 나가진 않았지만 부드럽고 예뻐요~~,5,"{'사이즈': '적당함', '색상': '같음', '소재': '같음'}",[]
4,308886954,틸아이다이,[단 하루!] [헤이가가 Pick]Wool cashmere V neck knit s...,https://www.wconcept.co.kr/Product/308886954,1,5,ms******,2026.09.10,IVORY_FREE,촉감도 좋고 다 좋은데 기장이 많이 짧아요 특히 이너 나시는 진짜 너무 짧고 작아요 ㅠㅠ,5,"{'사이즈': '작음', '색상': '같음', '소재': '같음'}",[https://media.wconcept.co.kr/review/308886954...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2613,300964307,비에이유 바이 브라이드앤유,[11차]ODETTE Square neck Three-quarter Sleeve H...,https://www.wconcept.co.kr/Product/300964307,3,17,bu******,2026.02.12,"1, BLACK",결혼식 피로연 원피스류 샀는데 딱 맞고 마음에 들어요! 원래 정장 바지 세트로 샀는...,5,"{'사이즈': '적당함', '색상': '같음', '소재': '같음'}",[https://media.wconcept.co.kr/review/300964307...
2614,300964307,비에이유 바이 브라이드앤유,[11차]ODETTE Square neck Three-quarter Sleeve H...,https://www.wconcept.co.kr/Product/300964307,3,18,ki******,2026.02.07,"3, BLACK",저에게는 팔 라인이 조금 애매한 것 같긴 하지만 격식있는 블랙원피스로 허리라인도 잘...,5,"{'사이즈': '적당함', '색상': '같음', '소재': '같음'}",[https://media.wconcept.co.kr/review/300964307...
2615,308839510,무니드,[단 하루!30%쿠폰] BREEZE BALLOON JUMPER - CHARCOAL,https://www.wconcept.co.kr/Product/308839510,1,1,wc******,2026.09.02,CHARCOAL O.S,비슷한 스타일 자켓이 많아서 구매 할까 말까 고민 많이했는데 역시나 구매 하길 잘했...,5,"{'사이즈': '적당함', '색상': '비슷함', '소재': '비슷함'}",[https://media.wconcept.co.kr/review/308839510...
2616,308839510,무니드,[단 하루!30%쿠폰] BREEZE BALLOON JUMPER - CHARCOAL,https://www.wconcept.co.kr/Product/308839510,1,2,wa******,2026.08.31,CHARCOAL O.S,하나 더 삽니다 동생이 마음에 든다고 입어서 저는 같이 입는건 싫어서 하나 더 사요,5,"{'사이즈': '적당함', '색상': '비슷함', '소재': '비슷함'}",[https://media.wconcept.co.kr/review/308839510...


In [ ]:
# 목적.목표 데이터 수집
# 해당 데이터 간단하게 보관.저장 (csv, jsonl) -> 빅데이터 취급.보관 한계 // 여러사람들이 동시에 접속해서 한 번에 같이 조회 x
# 한 저장소 안에 데이터를 담아놓고, 여러 사람들이 같이 공유해볼 수 있는 형태의 데이터 저장 필요!